# gradient-boosted tree: lgb

In [1]:
import pandas as pd

In [2]:
from dataengineers import Dataset

In [3]:
dataset = Dataset('train')
train, test = dataset.build_train_test()

/Users/matusbolecek/repos/nitor-comp/dataengineers.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df[f'{col}_rolling_mean_{window}h'] = (
/Users/matusbolecek/repos/nitor-comp/dataengineers.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df[f'{col}_ramp_{diff}h'] = (


In [4]:
exclude = ['id', 'target', 'delivery_start', 'market', 'market_int', 'timestamp_int', 'day_of_year', 'dayofweek', 'hour']

In [5]:
features = [c for c in train.columns if c not in exclude]

In [6]:
from models import lGBM

In [7]:
lg = lGBM(features)

In [8]:
lg.fit(train, test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006644 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23337
[LightGBM] [Info] Number of data points in the train set: 106086, number of used features: 99
[LightGBM] [Info] Start training from score 3.035028
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[335]	valid_0's rmse: 1.82006	valid_0's l2: 3.31261


In [9]:
y_vals = lg.predict(test)

In [10]:
from utils import rmse

In [11]:
rmse(test['target'], y_vals)

np.float64(49.568267979332)

In [12]:
ds2 = Dataset('test')

In [13]:
df_out = ds2.build_main()

/Users/matusbolecek/repos/nitor-comp/dataengineers.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df[f'{col}_rolling_mean_{window}h'] = (
/Users/matusbolecek/repos/nitor-comp/dataengineers.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df[f'{col}_ramp_{diff}h'] = (


In [14]:
y_out = lg.predict(df_out)

In [15]:
df_out['target'] = y_out

/var/folders/9p/v_hqlw5n24gdvttxlkst5c380000gn/T/ipykernel_40113/1019923952.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out['target'] = y_out


In [16]:
df_out.head()

,id,global_horizontal_irradiance,diffuse_horizontal_irradiance,direct_normal_irradiance,cloud_cover_total,cloud_cover_low,cloud_cover_mid,cloud_cover_high,precipitation_amount,visibility,...,renewables_lag_6h,renewables_lag_12h,renewables_lag_24h,renewables_lag_48h,renewables_rolling_mean_6h,renewables_rolling_mean_24h,renewables_ramp_1h,renewables_ramp_3h,renewables_ramp_6h,target
0,133627,0.0,0.0,0.0,100.0,14.0,44.0,100.0,0.0,16600.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.129338
1,133635,0.0,0.0,0.0,100.0,100.0,100.0,88.0,0.0,13800.0,...,NaN,NaN,NaN,NaN,3662.200000,3662.200000,-739.7,NaN,NaN,13.607788
2,133643,0.0,0.0,0.0,100.0,70.0,100.0,100.0,0.0,19700.0,...,NaN,NaN,NaN,NaN,3292.350000,3292.350000,-723.4,NaN,NaN,49.771715
3,133651,0.0,0.0,0.0,99.0,13.0,99.0,94.0,0.0,16200.0,...,NaN,NaN,NaN,NaN,2927.933333,2927.933333,-403.6,-1866.7,NaN,45.984114
4,133659,0.0,0.0,0.0,96.0,0.0,96.0,79.0,0.0,14500.0,...,NaN,NaN,NaN,NaN,2644.825000,2644.825000,-318.0,-1445.0,NaN,45.984114


In [17]:
from utils import Submission

In [18]:
submit = Submission(df_out)

In [19]:
submit.process()

,id,target
0,133627,53.129338
2183,133629,17.814064
4366,133630,22.636935
10915,133631,10.301070
6549,133633,5.283981
...,...,...
4365,146774,34.575239
6548,146775,14.461945
13097,146776,17.339211
8731,146777,25.247085


In [20]:
submit.validate()

✅ Validation passed!


In [21]:
submit.dump()